# 03 BKT baseline on Junyi (C1 Adaptive Curriculum Engine)

Fits Bayesian Knowledge Tracing per exercise on the training students only, then predicts the held-out test students.

1. **Fit** prior, learn, guess and slip per exercise with EM (Baum-Welch), training students only. Implemented in vectorised numpy: pyBKT's compiled extension does not build on Colab's Python 3.13, and its pure-Python fallback did not finish within 60 minutes even on 6,000 students.
2. **Check** for degenerate exercises (guess + slip >= 1), which are replaced by the median.
3. **Forward pass** with the same update formula as `curriculum-service/app/domain/mastery.py`.
4. **Cross-check against pyBKT** on a small subset: the service formula must reproduce pyBKT's predictions exactly, and our EM must fit at least as well as pyBKT's.
5. **Evaluate** on the test students: AUC (primary), accuracy, RMSE, against the majority-class baseline.
6. **Export** per-exercise parameters for notebook 04 (GAT features) and median values for the service defaults.

Needs notebooks 01 and 02. CPU is enough; the fit takes a few minutes. `QUICK = True` gives a short test run.

In [1]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"
# True: 3,000 training and 1,000 test students, for a fast check that everything runs.
QUICK = False
# Fit BKT on this many training students (random, seed 42); None = all 24,000.
FIT_STUDENTS = None

In [2]:
import sys
!{sys.executable} -m pip install -q pyBKT pyarrow "scikit-learn<1.8"

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "bkt"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert (PROCESSED / "interactions.parquet").exists(), "Run notebook 01 first (or upload its outputs)."
print("Storage:", STORAGE, "->", BASE)

Mounted at /content/drive
Storage: drive -> /content/drive/MyDrive/adaptlearn-c1


## 1. Load the answers and the student split from notebook 01

In [3]:
import json
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

inter = pd.read_parquet(PROCESSED / "interactions.parquet",
                        columns=["user_id", "exercise", "exercise_idx", "step", "correct"])
splits = json.loads((PROCESSED / "splits.json").read_text())
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())

train_ids = np.array(sorted(u for fold in splits["folds"].values() for u in fold))
test_ids = np.array(splits["test"])
if QUICK:
    train_ids = rng.choice(train_ids, 3_000, replace=False)
    test_ids = rng.choice(test_ids, 1_000, replace=False)

# Sorted by student, then time; order_id is a global running number pyBKT can sort on.
inter = inter.sort_values(["user_id", "step"], kind="mergesort").reset_index(drop=True)
inter["order_id"] = np.arange(len(inter))
fit_ids = train_ids
if FIT_STUDENTS and not QUICK and FIT_STUDENTS < len(train_ids):
    fit_ids = rng.choice(train_ids, FIT_STUDENTS, replace=False)
train = inter[inter.user_id.isin(fit_ids)].reset_index(drop=True)
test = inter[inter.user_id.isin(test_ids)].reset_index(drop=True)
assert not set(train.user_id) & set(test.user_id), "a test student is in training"
print(f"train: {train.user_id.nunique()} students, {len(train):,} answers")
print(f"test:  {test.user_id.nunique()} students, {len(test):,} answers")

train: 24000 students, 4,781,106 answers
test:  6000 students, 1,178,267 answers


## 2. Fit BKT per exercise with EM (training students only)

Every (student, exercise) pair is one sequence of answers. BKT is a two-state hidden Markov model (not learned / learned, no forgetting), so its parameters are fitted with Baum-Welch EM:

* **E-step**: a scaled forward-backward pass gives, for every answer, the probability the student had learned the exercise at that moment, and the expected number of not-learned to learned transitions.
* **M-step**: prior = mean probability of learned at the first answer; learn = expected transitions / expected time not learned; guess = correct answers while not learned / time not learned; slip = wrong answers while learned / time learned.

All exercises are fitted at once: sequences are sorted longest first and stored time-major, so step *t* of every sequence still running is one contiguous array slice. Same start values for every exercise (prior 0.3, learn 0.1, guess 0.2, slip 0.1); stop when no parameter moves more than 1e-5.

In [4]:
import time

EM_INIT = (0.3, 0.1, 0.2, 0.1)  # prior, learn, guess, slip
EM_MAX_ITER, EM_TOL, EPS = 200, 1e-5, 1e-6
all_exercises = sorted(exercise_index, key=exercise_index.get)
N_EX = len(all_exercises)


def build_sequences(df: pd.DataFrame) -> dict:
    """One sequence per (student, exercise), stored time-major, longest sequences first."""
    key = df["user_id"].astype(str) + "|" + df["exercise_idx"].astype(str)
    seq_id, _ = pd.factorize(key, sort=False)
    pos = df.groupby(seq_id, sort=False).cumcount().to_numpy()  # df is sorted by student, time
    lengths = np.bincount(seq_id)
    order = np.argsort(-lengths, kind="stable")
    rank = np.empty_like(order)
    rank[order] = np.arange(len(order))
    tm = np.lexsort((rank[seq_id], pos))
    counts = np.bincount(pos)  # sequences still running at step t
    seq_ex = np.empty(len(lengths), dtype=np.int64)
    seq_ex[seq_id] = df["exercise_idx"].to_numpy()
    return {"y": df["correct"].to_numpy().astype(bool)[tm], "ex": seq_ex[order], "counts": counts,
            "off": np.concatenate([[0], np.cumsum(counts)])}


def forward(S, p0, l, g, s):
    """Scaled forward pass: P(learned), P(not learned) after each answer, and P(answer | earlier answers)."""
    y, ex, counts, off = S["y"], S["ex"], S["counts"], S["off"]
    aL, aU, c = np.empty(len(y)), np.empty(len(y)), np.empty(len(y))
    for t in range(len(counts)):
        k, sl = counts[t], slice(off[t], off[t + 1])
        e, yt = ex[:k], y[sl]
        if t == 0:
            predL, predU = p0[e], 1 - p0[e]
        else:
            pL, pU = aL[off[t - 1]:off[t - 1] + k], aU[off[t - 1]:off[t - 1] + k]
            predL, predU = pL + pU * l[e], pU * (1 - l[e])
        fL = predL * np.where(yt, 1 - s[e], s[e])
        fU = predU * np.where(yt, g[e], 1 - g[e])
        c[sl] = fL + fU
        aL[sl], aU[sl] = fL / c[sl], fU / c[sl]
    return aL, aU, c


def em_bkt(S, verbose=True):
    """Baum-Welch EM for BKT, every exercise at once. Returns parameters per exercise, log-likelihood trace, iterations."""
    y, ex, counts, off = S["y"], S["ex"], S["counts"], S["off"]
    T, N = len(counts), len(y)
    step = np.repeat(np.arange(T), counts)
    ex_all = ex[np.arange(N) - off[step]]  # exercise of every answer
    used = np.bincount(ex, minlength=N_EX) > 0
    p0, l, g, s = (np.full(N_EX, v) for v in EM_INIT)
    trace = []
    for it in range(EM_MAX_ITER):
        aL, aU, c = forward(S, p0, l, g, s)
        trace.append(float(np.log(c).sum()))
        # Backward pass with the same scaling; gL = P(learned at t | all answers of the sequence).
        gL = np.empty(N)
        xi_UL, time_U = np.zeros(N_EX), np.zeros(N_EX)
        bL_next = bU_next = None
        for t in range(T - 1, -1, -1):
            k, sl = counts[t], slice(off[t], off[t + 1])
            bL, bU = np.ones(k), np.ones(k)
            if t + 1 < T:
                k2, sl2 = counts[t + 1], slice(off[t + 1], off[t + 2])
                e2, y2 = ex[:k2], y[sl2]
                wL = np.where(y2, 1 - s[e2], s[e2]) * bL_next / c[sl2]
                wU = np.where(y2, g[e2], 1 - g[e2]) * bU_next / c[sl2]
                bL[:k2], bU[:k2] = wL, (1 - l[e2]) * wU + l[e2] * wL
                aU_t = aU[off[t]:off[t] + k2]
                xi_UL += np.bincount(e2, weights=aU_t * l[e2] * wL, minlength=N_EX)
                time_U += np.bincount(e2, weights=aU_t * bU[:k2], minlength=N_EX)
            gL[sl] = aL[sl] * bL
            bL_next, bU_next = bL, bU
        gU = 1 - gL
        first = ex[:counts[0]]
        new = (
            np.bincount(first, weights=gL[:counts[0]], minlength=N_EX)
            / np.maximum(np.bincount(first, minlength=N_EX), 1),
            xi_UL / np.maximum(time_U, EPS),
            np.bincount(ex_all, weights=gU * y, minlength=N_EX)
            / np.maximum(np.bincount(ex_all, weights=gU, minlength=N_EX), EPS),
            np.bincount(ex_all, weights=gL * ~y, minlength=N_EX)
            / np.maximum(np.bincount(ex_all, weights=gL, minlength=N_EX), EPS),
        )
        new = [np.clip(v, EPS, 1 - EPS) for v in new]
        change = max(float(np.abs(a - b)[used].max()) for a, b in zip(new, (p0, l, g, s)))
        p0, l, g, s = new
        if verbose and it % 10 == 0:
            print(f"iteration {it:3d}  log-likelihood {trace[-1]:,.1f}  largest change {change:.1e}")
        if change < EM_TOL:
            break
    out = pd.DataFrame({"p_init": p0, "p_learn": l, "p_guess": g, "p_slip": s}, index=all_exercises)
    return out[used], trace, it + 1


start = time.time()
S_train = build_sequences(train)
params, em_trace, em_iterations = em_bkt(S_train)
assert all(b >= a - 1e-6 * abs(a) for a, b in zip(em_trace, em_trace[1:])), "EM log-likelihood went down"
print(f"fitted {len(params)} exercises on {train.user_id.nunique():,} students in "
      f"{(time.time() - start) / 60:.1f} min, {em_iterations} iterations")
params.index.name = "skill"
params.describe().round(3)

iteration   0  log-likelihood -2,054,456.8  largest change 8.0e-01
iteration  10  log-likelihood -1,771,522.2  largest change 4.9e-02
iteration  20  log-likelihood -1,770,939.5  largest change 2.2e-02
iteration  30  log-likelihood -1,770,804.4  largest change 3.1e-02
iteration  40  log-likelihood -1,770,760.3  largest change 1.5e-02
iteration  50  log-likelihood -1,770,727.8  largest change 2.1e-02
iteration  60  log-likelihood -1,770,713.4  largest change 1.3e-02
iteration  70  log-likelihood -1,770,707.1  largest change 9.9e-03
iteration  80  log-likelihood -1,770,703.2  largest change 9.3e-03
iteration  90  log-likelihood -1,770,700.4  largest change 1.0e-02
iteration 100  log-likelihood -1,770,698.2  largest change 1.1e-02
iteration 110  log-likelihood -1,770,696.4  largest change 9.7e-03
iteration 120  log-likelihood -1,770,694.9  largest change 8.0e-03
iteration 130  log-likelihood -1,770,693.7  largest change 6.1e-03
iteration 140  log-likelihood -1,770,692.7  largest change 4.3

,p_init,p_learn,p_guess,p_slip
count,683.000,683.000,683.000,683.000
mean,0.453,0.153,0.283,0.129
std,0.275,0.203,0.203,0.066
min,0.000,0.000,0.000,0.000
25%,0.231,0.040,0.116,0.080
50%,0.440,0.081,0.263,0.126
75%,0.695,0.169,0.407,0.170
max,0.995,1.000,1.000,0.460


## 3. Degenerate exercises

When guess + slip >= 1, a correct answer is more likely from a student who has **not** learned the exercise, so the model runs backwards. Those exercises get the median of the healthy ones. Exercises with no training answers get the median too.

In [5]:
healthy = params[(params.p_guess + params.p_slip) < 1]
median = healthy.median()
degenerate = sorted(set(params.index) - set(healthy.index))
print(f"degenerate exercises: {len(degenerate)} of {len(params)}")

final = params.copy()
final.loc[degenerate] = median.values
final = final.reindex(all_exercises)
unseen = final.p_init.isna().sum()
final = final.fillna(median)
final["status"] = "fitted"
final.loc[final.index.isin(degenerate), "status"] = "median (degenerate)"
final.loc[~final.index.isin(params.index), "status"] = "median (no training answers)"
final.index.name = "exercise"
print(f"exercises without training answers: {unseen}")
final.status.value_counts()

degenerate exercises: 3 of 683
exercises without training answers: 93


,count
status,
fitted,680
median (no training answers),93
median (degenerate),3


## 4. Forward pass with the service's formula

The same two steps as `app/domain/mastery.py`: Bayes' rule on the answer, then the learning step (every Junyi answer is practice). `mastery_before` is the probability the exercise is learned **before** the answer, so the prediction for answer *t* only uses answers before *t*.

In [6]:
def bkt_forward(df: pd.DataFrame, p: pd.DataFrame) -> pd.DataFrame:
    ex = df["exercise_idx"].to_numpy()
    users = df["user_id"].to_numpy()
    y = df["correct"].to_numpy().astype(bool)
    by_idx = p.reindex(all_exercises)
    prior, learn = by_idx.p_init.to_numpy(), by_idx.p_learn.to_numpy()
    guess, slip = by_idx.p_guess.to_numpy(), by_idx.p_slip.to_numpy()

    before = np.empty(len(df))
    state = prior.copy()
    for i in range(len(df)):
        if i == 0 or users[i] != users[i - 1]:
            state = prior.copy()
        e = ex[i]
        m = state[e]
        before[i] = m
        if y[i]:
            post = m * (1 - slip[e]) / (m * (1 - slip[e]) + (1 - m) * guess[e])
        else:
            post = m * slip[e] / (m * slip[e] + (1 - m) * (1 - guess[e]))
        state[e] = post + (1 - post) * learn[e]
    p_correct = before * (1 - slip[ex]) + (1 - before) * guess[ex]
    return pd.DataFrame({"mastery_before": before, "p_correct": p_correct})

start = time.time()
pred = bkt_forward(test, final)
print(f"forward pass over {len(test):,} answers in {time.time() - start:.0f} s")

forward pass over 1,178,267 answers in 2 s


### Cross-check against pyBKT

pyBKT is the reference BKT library. On a small subset (300 training and 300 test students, the 5 most answered exercises) it checks two things: (a) the service's update formula reproduces pyBKT's predictions from pyBKT's own parameters, and (b) our EM reaches at least pyBKT's log-likelihood. Parameters can differ slightly between two EM runs (different start values, local optima), so the comparison is on fit quality, not on equal numbers.

In [7]:
# (a) The service formula must reproduce pyBKT's predictions from pyBKT's own parameters.
# (b) Our EM must fit the subset at least as well as pyBKT's EM (log-likelihood, test AUC).
from pyBKT.models import Model
from sklearn.metrics import roc_auc_score

# pyBKT is unmaintained and breaks on numpy >= 2.4 (Colab ships 2.0.x).
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2, 4), (
    "Run: !pip install -q numpy==2.0.2   then Runtime > Restart session, and run again.")


def as_pybkt(df):
    # pyBKT's default column names; predict() does not accept a column mapping.
    return df.rename(columns={"exercise": "skill_name"})[["order_id", "user_id", "skill_name", "correct"]]


def full(p):
    return p.reindex(all_exercises).fillna(0.5)


CHECK_EXERCISES = train.exercise.value_counts().index[:5].tolist()
check_rng = np.random.default_rng(SEED)
check_train = train[train.user_id.isin(check_rng.choice(train.user_id.unique(), 300, replace=False))
                    & train.exercise.isin(CHECK_EXERCISES)].reset_index(drop=True)
check_test = test[test.user_id.isin(check_rng.choice(test.user_id.unique(), 300, replace=False))
                  & test.exercise.isin(CHECK_EXERCISES)].reset_index(drop=True)
print(f"subset: {len(CHECK_EXERCISES)} exercises, {len(check_train):,} training and {len(check_test):,} test answers")

start = time.time()
pybkt = Model(seed=SEED, num_fits=1)
pybkt.fit(data=as_pybkt(check_train))
raw = pybkt.params().reset_index()
pybkt_params = (raw[raw["param"].isin(["prior", "learns", "guesses", "slips"])]
                .pivot_table(index="skill", columns="param", values="value", aggfunc="first")
                .rename(columns={"prior": "p_init", "learns": "p_learn", "guesses": "p_guess", "slips": "p_slip"})
                [["p_init", "p_learn", "p_guess", "p_slip"]])
print(f"pyBKT fit in {time.time() - start:.0f} s")
ours_params, _, _ = em_bkt(build_sequences(check_train), verbose=False)

# (a) Formula: our forward pass with pyBKT's parameters vs pyBKT.predict.
pybkt_pred = pybkt.predict(data=as_pybkt(check_test)).sort_values("order_id")["correct_predictions"].to_numpy()
own = bkt_forward(check_test, full(pybkt_params))["p_correct"].to_numpy()
diff = np.abs(own - pybkt_pred)
print(f"(a) max |service formula - pyBKT| = {diff.max():.2e}")
print("    MATCH: the service formula equals pyBKT" if diff.max() < 1e-4
      else "    MISMATCH: send this output to Claude before going on")

# (b) Fit quality on the same subset.
S_check = build_sequences(check_train)
fits = {"ours": ours_params, "pyBKT": pybkt_params}
ll = {name: float(np.log(forward(S_check, *(full(p)[c].to_numpy() for c in p.columns))[2]).sum())
      for name, p in fits.items()}
auc = {name: float(roc_auc_score(check_test["correct"], bkt_forward(check_test, full(p))["p_correct"]))
       for name, p in fits.items()}
pybkt_check = {"exercises": CHECK_EXERCISES, "train_answers": len(check_train), "test_answers": len(check_test),
               "max_formula_diff": float(diff.max()), "log_likelihood": ll, "test_auc": auc}
print(f"(b) training log-likelihood  ours {ll['ours']:,.1f}  pyBKT {ll['pyBKT']:,.1f}  (higher is better)")
print(f"    test AUC                 ours {auc['ours']:.4f}  pyBKT {auc['pyBKT']:.4f}")
print("    OK: our EM fits at least as well as pyBKT" if ll["ours"] >= ll["pyBKT"] - 1e-3 * abs(ll["pyBKT"])
      else "    CHECK: pyBKT fits better; send this output to Claude")
pd.concat(fits, axis=1).round(3)

subset: 5 exercises, 9,180 training and 7,880 test answers


/usr/local/lib/python3.13/dist-packages/pyBKT/fit/EM_fit.py:227: RuntimeWarning: invalid value encountered in divide
  pair[:, 0] = (pair[:, 0] * gamma_t) / dotted
/usr/local/lib/python3.13/dist-packages/pyBKT/fit/EM_fit.py:228: RuntimeWarning: invalid value encountered in divide
  pair[:, 1] = (pair[:, 1] * gamma_t) / dotted
/usr/local/lib/python3.13/dist-packages/pyBKT/fit/EM_fit.py:227: RuntimeWarning: invalid value encountered in divide
  pair[:, 0] = (pair[:, 0] * gamma_t) / dotted
/usr/local/lib/python3.13/dist-packages/pyBKT/fit/EM_fit.py:228: RuntimeWarning: invalid value encountered in divide
  pair[:, 1] = (pair[:, 1] * gamma_t) / dotted
/usr/local/lib/python3.13/dist-packages/pyBKT/fit/EM_fit.py:227: RuntimeWarning: invalid value encountered in divide
  pair[:, 0] = (pair[:, 0] * gamma_t) / dotted
/usr/local/lib/python3.13/dist-packages/pyBKT/fit/EM_fit.py:228: RuntimeWarning: invalid value encountered in divide
  pair[:, 1] = (pair[:, 1] * gamma_t) / dotted
/usr/local/lib/p

pyBKT fit in 48 s
(a) max |service formula - pyBKT| = 2.11e-15
    MATCH: the service formula equals pyBKT
(b) training log-likelihood  ours -2,010.5  pyBKT -2,020.7  (higher is better)
    test AUC                 ours 0.6695  pyBKT 0.6620
    OK: our EM fits at least as well as pyBKT


ours                           pyBKT                  \
                      p_init p_learn p_guess  p_slip  p_init p_learn p_guess   
addition_1           0.15800 0.00400 0.98800 0.07000 0.28400 0.03400 1.00000   
number_line          0.76600 0.47300 0.65900 0.04200 0.34400 0.81500 0.84000   
representing_numbers 0.89400 0.38700 0.10600 0.08800 0.88500 0.39600 0.15700   
subtraction_1        0.35100 0.07100 0.99800 0.05400 0.40300 0.07200 1.00000   
telling_time_0.5     0.86600 0.19300 0.28100 0.08700 0.86100 0.19800 0.28400   

                              
                      p_slip  
addition_1           0.05500  
number_line          0.04500  
representing_numbers 0.08600  
subtraction_1        0.05300  
telling_time_0.5     0.08700

## 5. Results on the held-out test students

In [8]:
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score

y_test = test["correct"].to_numpy()
majority = int(train["correct"].mean() >= 0.5)
results = pd.DataFrame([
    {"model": "majority class", "auc": 0.5,
     "accuracy": accuracy_score(y_test, np.full_like(y_test, majority)),
     "rmse": np.sqrt(mean_squared_error(y_test, np.full(len(y_test), train["correct"].mean())))},
    {"model": "BKT (EM per exercise)", "auc": roc_auc_score(y_test, pred.p_correct),
     "accuracy": accuracy_score(y_test, pred.p_correct >= 0.5),
     "rmse": np.sqrt(mean_squared_error(y_test, pred.p_correct))},
]).round(4)
results

,model,auc,accuracy,rmse
0,majority class,0.50000,0.83420,0.37190
1,BKT (EM per exercise),0.78040,0.85100,0.33590


## 6. Defaults for the service

The service starts every Java concept from one set of BKT values (it has no data of its own yet). Prior, learn and slip are the medians over the healthy Junyi exercises. Guess stays at 0.25 in the service, because every C1 item is a four-option multiple choice, unlike Junyi's open answers.

In [9]:
service_defaults = {
    "p_init": round(float(median.p_init), 4),
    "p_learn": round(float(median.p_learn), 4),
    "p_slip": round(float(median.p_slip), 4),
    "p_guess_junyi_median": round(float(median.p_guess), 4),
    "p_guess_service": 0.25,
    "source": "median over healthy Junyi exercises, training students only",
}
print(json.dumps(service_defaults, indent=2))
print("\nPaste into app/domain/mastery.py:")
print(f'DEFAULT_PARAMS = BktParams(p_init={service_defaults["p_init"]}, p_learn={service_defaults["p_learn"]}, '
      f'p_guess=0.25, p_slip={service_defaults["p_slip"]}, version="bkt-junyi-v1")')

{
  "p_init": 0.4404,
  "p_learn": 0.0801,
  "p_slip": 0.1268,
  "p_guess_junyi_median": 0.2595,
  "p_guess_service": 0.25,
  "source": "median over healthy Junyi exercises, training students only"
}

Paste into app/domain/mastery.py:
DEFAULT_PARAMS = BktParams(p_init=0.4404, p_learn=0.0801, p_guess=0.25, p_slip=0.1268, version="bkt-junyi-v1")


## 7. Save outputs and a manifest

In [10]:
import platform
import sklearn
import pyBKT

final.to_csv(OUT_DIR / "params.csv")
(OUT_DIR / "service_defaults.json").write_text(json.dumps(service_defaults, indent=2))
(OUT_DIR / "metrics.json").write_text(results.to_json(orient="records", indent=2))
(OUT_DIR / "degenerate.json").write_text(json.dumps(degenerate))
(OUT_DIR / "pybkt_check.json").write_text(json.dumps(pybkt_check, indent=2))

CONFIG = {"SEED": SEED, "QUICK": QUICK, "fit": "numpy Baum-Welch EM", "forgets": False,
          "em_init": EM_INIT, "em_tol": EM_TOL, "em_iterations": em_iterations,
          "em_log_likelihood": em_trace[-1], "pybkt_check": pybkt_check,
          "train_students": int(len(train_ids)), "fit_students": int(len(fit_ids)), "test_students": int(len(test_ids))}
VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "scikit-learn": sklearn.__version__, "pyBKT": getattr(pyBKT, "__version__", "unknown")}

In [11]:
import hashlib, json, platform
from datetime import datetime, timezone

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "versions": VERSIONS,
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.is_file() and f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

{
  "created_at": "2026-10-09T10:23:22.902018+00:00",
  "config": {
    "SEED": 42,
    "QUICK": false,
    "fit": "numpy Baum-Welch EM",
    "forgets": false,
    "em_init": [
      0.3,
      0.1,
      0.2,
      0.1
    ],
    "em_tol": 1e-05,
    "em_iterations": 200,
    "em_log_likelihood": -1770688.6634753582,
    "pybkt_check": {
      "exercises": [
        "addition_1",
        "number_line",
        "representing_numbers",
        "telling_time_0.5",
        "subtraction_1"
      ],
      "train_answers": 9180,
      "test_answers": 7880,
      "max_formula_diff": 2.1094237467877974e-15,
      "log_likelihood": {
        "ours": -2010.4588564833225,
        "pyBKT": -2020.7370289392989
      },
      "test_auc": {
        "ours": 0.6694807783659066,
        "pyBKT": 0.6619911483781217
      }
    },
    "train_students": 24000,
    "fit_students": 24000,
    "test_students": 6000
  },
  "versions": {
    "python": "3.13.15",
    "pandas": "2.2.3",
    "numpy": "2.1.3",
    

In [12]:
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive("/content/bkt_outputs", "zip", OUT_DIR))
else:
    print("Outputs stored in", OUT_DIR)

Outputs stored in /content/drive/MyDrive/adaptlearn-c1/junyi_processed/bkt


## For the report

Report the results table (AUC is the primary metric; accuracy must be read against the 83.2% majority class), the number of degenerate exercises, the pyBKT cross-check (formula match, log-likelihood and AUC on the subset) and the service defaults. Method sentence: *BKT parameters were fitted per exercise with Baum-Welch EM implemented in numpy and verified against pyBKT on a subset.* Notebook 04 reads `bkt/params.csv`.

Before committing: **Clear all outputs**.